**This notebook contains the source code for the road anomaly detection model. Its content runs right from obtaining the dataset to evaluating the performance of the trained model.**

# Model Training

This section contains the training pipeline for the YOLOv9 extended model. Availability of the dataset in the current working directory and in proper split has been assumed.

To obtain the cleaned and structured dataset used for training, go to [Kaggle](https://www.kaggle.com/datasets/david2do/road-anomaly-ds) to download it.

In [1]:
# Install and import packages

!pip install ultralytics
from ultralytics import YOLO

In [ ]:
# Load the yolov9e architecture, and initialize with COCO pre-trained weights

model = YOLO('yolov9e.pt')

In [ ]:
# Training parameters

data = './data.yaml'     # Assuming the yaml file is in the current working directory
epochs = 100             #  Number of epochs
batch_size = 16          # Batch size to use
lr = 0.1                 # learning rate (modify for the five learning rates)
img_size = 640           # input image size for training
optimizer = 'AdamW'      # Optimizer for model training

# Augmentation parameters for mosaic, mixup and cutmix
mosaic = 1.0          
mixup = 0.5
cutmix = 0.5

In [ ]:
# Model Training with Ultralytics

model.train(
    data=data,
    epochs=epochs,
    batch=batch_size,
    lr0=lr,
    imgsz=img_size,
    optimizer=optimizer,
    workers=8,                 # Number of CPU workers to load dataset
    amp=False,                 # Disable Automatic Mixed Precision
    project='./runs',          # Directory to save training logs
    name='yolov9_lr_0.1',      # Sub-directory to save training logs
    mosaic=mosaic,
    mixup=mixup,
    cutmix=cutmix
)

# Model Performance Evaluation and Testing

This section contains code for evaluating the performance of the trained model on the **test dataset**. Output includes metrics such as mAP@50, mAP@50:95, Precision, Recall, and their plots. Additionally, the model's performance on each class was obtained.

Inference performed on six samples of the test set is also displayed to ascertain correct predictions.

In [2]:
# Install and import necessary packages

!pip install ultralytics
from ultralytics import YOLO

In [ ]:
# Load trained model

weight_file = 'best_lr_0.1.pt'      # using the best weight of lr at 0.1 

model = YOLO(weight_file)

In [ ]:
# Run evaluation on test set
# This outputs all, and per-class metrics.

results = model.val(data="./data.yaml", split="test")

In [ ]:
# Plot predictions on random six test set images

image_folder = "./test/images"  # Test set folder containing images
num_images = 6                 # Number of random images to plot

all_images = [os.path.join(image_folder, f) 
              for f in os.listdir(image_folder) 
              if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

random_images = random.sample(all_images, min(num_images, len(all_images)))

plt.figure(figsize=(15, 10))

for i, img_path in enumerate(random_images, 1):

    results = model.predict(source=img_path, save=False, verbose=False)
    img_with_boxes = results[0].plot()

    img_rgb = cv2.cvtColor(img_with_boxes, cv2.COLOR_BGR2RGB)

    # Plot
    plt.subplot(2, 3, i)
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(os.path.basename(img_path))

plt.tight_layout()
plt.show()